# 04 — Functional Java & Streams

**What you'll learn:**

- Lambdas — the arrow syntax and what they replace
- Method references — `Class::method` shorthand
- The four core functional interfaces: `Function`, `Predicate`, `Consumer`, `Supplier`
- Two-argument variants: `BiFunction`, `BiPredicate`, `BiConsumer`
- Writing your own functional interface with `@FunctionalInterface`
- How lambdas capture surrounding variables — the *effectively final* rule
- The `Stream<T>` pipeline model — lazy, single-use, three stages
- Creating streams: from collections, `Stream.of`, `IntStream.range`, `Stream.generate`, `Stream.iterate`
- Intermediate operations: `filter`, `map`, `flatMap`, `distinct`, `sorted`, `peek`, `limit`, `skip`
- Terminal operations: `collect`, `reduce`, `forEach`, `count`, `anyMatch`, `findFirst`
- Collectors: `toList`, `toSet`, `toMap`, `groupingBy`, `partitioningBy`, `joining`
- Parallel streams — when they help and when they hurt
- `Optional<T>` — the type-safe alternative to `null`
- `Optional` operations: `map`, `flatMap`, `filter`, `orElse`, `orElseGet`, `orElseThrow`, `ifPresent`

Notebook 03 gave us collections. This notebook turns those collections into *pipelines* — the way modern Java code actually iterates, filters, transforms, and aggregates data.

## Lambdas — functions as values

A **lambda expression** is a short, inline function you can pass around like any other value. Before Java 8, the way to pass behaviour to a method was an anonymous inner class — three lines of ceremony for one expression of code. Lambdas reduce that to one line.

The syntax is `parameters -> body`. Pick whichever form fits:

```text
   ()         -> 42                  // no params
   x          -> x * x               // one param, no parens needed
   (x, y)     -> x + y               // multiple params
   (int x)    -> x + 1               // optional explicit types
   x -> { var d = x * 2; return d; } // block body with explicit return
```

A lambda is a value of some **functional interface** type — an interface with exactly one abstract method. The compiler infers which interface from the surrounding context.

In [ ]:
import java.util.*;
import java.util.function.*;

// before lambdas — anonymous inner class
Comparator<String> byLengthOld = new Comparator<String>() {
    @Override
    public int compare(String a, String b) {
        return Integer.compare(a.length(), b.length());
    }
};

// with a lambda — same thing, one line
Comparator<String> byLength = (a, b) -> Integer.compare(a.length(), b.length());

var words = new ArrayList<>(List.of("kiwi", "fig", "blueberry", "apple"));
words.sort(byLength);
// [fig, kiwi, apple, blueberry]

## Method references

When a lambda does nothing but call an existing method, you can replace it with a **method reference** — the `Class::method` (or `instance::method`) form. It's a one-to-one shorthand and reads cleaner than a forwarding lambda.

Four flavours:

| Form | Meaning |
|---|---|
| `Class::staticMethod` | reference to a static method |
| `instance::instanceMethod` | reference to a method on a specific instance |
| `Class::instanceMethod` | reference to an instance method, with the receiver as the first arg |
| `Class::new` | constructor reference |

In [ ]:
// these four lambdas all have method-reference equivalents
Function<String, Integer> parse1 = s -> Integer.parseInt(s);
Function<String, Integer> parse2 = Integer::parseInt;        // Class::staticMethod

var prefix = "user:";
Function<String, String> prepend1 = s -> prefix.concat(s);
Function<String, String> prepend2 = prefix::concat;          // instance::instanceMethod

Function<String, Integer> len1 = s -> s.length();
Function<String, Integer> len2 = String::length;             // Class::instanceMethod

Supplier<ArrayList<String>> newList1 = () -> new ArrayList<>();
Supplier<ArrayList<String>> newList2 = ArrayList::new;       // Class::new

## The four core functional interfaces

The `java.util.function` package ships ~40 functional interfaces, but four cover the vast majority of real code. Memorise the shape of each — they show up in every Stream operation.

| Interface | Method | Shape | Use |
|---|---|---|---|
| `Function<T,R>` | `R apply(T t)` | T → R | transform |
| `Predicate<T>` | `boolean test(T t)` | T → boolean | filter |
| `Consumer<T>` | `void accept(T t)` | T → () | side effect |
| `Supplier<T>` | `T get()` | () → T | produce |

Two-argument variants exist too: `BiFunction<T,U,R>`, `BiPredicate<T,U>`, `BiConsumer<T,U>`. And there are primitive-specialised forms — `IntFunction<R>`, `ToIntFunction<T>`, `IntPredicate` — that avoid boxing when working with `int`, `long`, or `double`.

In [ ]:
// Function — transform
Function<String, Integer> length = String::length;
length.apply("hello");           // 5

// Predicate — test
Predicate<String> isBlank = String::isBlank;
isBlank.test("");                // true
isBlank.test("hello");           // false

// Consumer — side effect, returns nothing
Consumer<String> println = System.out::println;
println.accept("hi");            // prints "hi"

// Supplier — produce a value on demand
Supplier<List<String>> empty = List::of;
empty.get();                     // []

// BiFunction — two-argument transform
BiFunction<Integer, Integer, Integer> add = Integer::sum;
add.apply(2, 3);                 // 5

## Writing your own functional interface

Any interface with exactly one abstract method is automatically usable as a lambda target. The `@FunctionalInterface` annotation documents that intent and makes the compiler enforce the *single abstract method* rule.

In [ ]:
@FunctionalInterface
interface ScoreFn {
    int score(String input);

    // default methods are allowed — they don't count toward the SAM rule
    default int doubled(String input) { return score(input) * 2; }
}

ScoreFn lengthScore = s -> s.length();
lengthScore.score("hello");      // 5
lengthScore.doubled("hello");    // 10

## Capturing variables — *effectively final*

A lambda can reference local variables from the enclosing scope, but only if those variables are **effectively final** — assigned exactly once. The compiler will reject a lambda that reads a variable you reassign later, or assigns to one inside the lambda body.

This restriction exists because lambdas can outlive the method that created them (think: pass a lambda to an executor that runs it on another thread). A capturing lambda holds a *snapshot* of the value at capture time; allowing reassignment would create confusing inconsistencies.

In [ ]:
var prefix = "user:";                            // effectively final — OK
Function<String, String> prepend = s -> prefix + s;
prepend.apply("ganesh");                         // "user:ganesh"

// var counter = 0;
// Runnable r = () -> counter++;    // COMPILE ERROR — would reassign captured variable

// The standard workaround: use a mutable single-element holder
var counter = new int[]{0};                      // array itself is effectively final
Runnable bump = () -> counter[0]++;              // mutating array contents is fine
bump.run(); bump.run(); bump.run();
counter[0];                                      // 3

## Streams — pipelines over collections

A `Stream<T>` is a *one-time-use* pipeline that pulls elements through a chain of operations. It is not a collection — it does not store the elements. It is a description of work to do, applied lazily when a terminal operation kicks off the pipeline.

```text
   source ──> intermediate ──> intermediate ──> terminal
   List      .filter(...)      .map(...)       .toList()
              \____________________/   \_______/
                 lazy, return Stream     eager, returns a value
```

Three stages:

1. **Source** — where the elements come from. Usually `collection.stream()`.
2. **Intermediate operations** — `filter`, `map`, etc. They return a new `Stream` and do nothing until a terminal op runs. Lazy.
3. **Terminal operation** — `collect`, `count`, `forEach`, etc. Triggers the pipeline and produces a result (or a side effect). Eager.

A stream can be traversed exactly once. After a terminal op, it's spent — any further operation throws.

## Creating streams

Streams come from many sources. The four you'll see most:

In [ ]:
import java.util.stream.*;

// 1. from a collection — most common
Stream<String> a = List.of("alice", "bob", "carol").stream();

// 2. Stream.of — literal sequence
Stream<Integer> b = Stream.of(1, 2, 3, 4, 5);

// 3. IntStream.range — numeric range (exclusive)
IntStream c = IntStream.range(0, 5);     // 0, 1, 2, 3, 4
IntStream d = IntStream.rangeClosed(1, 5);  // 1, 2, 3, 4, 5

// 4. infinite streams — must be bounded with limit()
Stream<Integer> e = Stream.iterate(1, n -> n * 2).limit(10);     // 1,2,4,8,...,512
Stream<Double>  f = Stream.generate(Math::random).limit(5);

## Intermediate operations

Intermediate operations transform one stream into another. They are **lazy** — calling them just records the operation; nothing runs until a terminal op pulls elements through.

| Operation | What it does |
|---|---|
| `filter(Predicate)` | keep elements that match |
| `map(Function)` | transform each element |
| `flatMap(Function)` | flatten nested streams |
| `distinct()` | remove duplicates |
| `sorted()` / `sorted(Comparator)` | sort |
| `limit(n)` / `skip(n)` | take or drop a prefix |
| `peek(Consumer)` | side-effect inspection — debug only |

In [ ]:
var nums = List.of(1, 2, 3, 4, 5, 6, 7, 8, 9, 10);

// filter + map + collect — the canonical pipeline
List<Integer> evenSquared = nums.stream()
    .filter(n -> n % 2 == 0)
    .map(n -> n * n)
    .toList();
// [4, 16, 36, 64, 100]

// flatMap — flatten a Stream<Stream<T>> into Stream<T>
var lines = List.of("the quick brown", "fox jumps over");
List<String> words = lines.stream()
    .flatMap(line -> Arrays.stream(line.split(" ")))
    .toList();
// [the, quick, brown, fox, jumps, over]

// distinct + sorted
List<Integer> uniqueSorted = Stream.of(3, 1, 4, 1, 5, 9, 2, 6, 5)
    .distinct()
    .sorted()
    .toList();
// [1, 2, 3, 4, 5, 6, 9]

## Terminal operations

Terminal operations trigger the pipeline and produce a result.

| Operation | Returns |
|---|---|
| `toList()` / `collect(...)` | a collection |
| `count()` | `long` count |
| `forEach(Consumer)` | nothing — side effect |
| `reduce(...)` | aggregated value |
| `anyMatch` / `allMatch` / `noneMatch` | `boolean` |
| `findFirst()` / `findAny()` | `Optional<T>` |
| `min(Comparator)` / `max(Comparator)` | `Optional<T>` |

In [ ]:
var nums = List.of(1, 2, 3, 4, 5);

nums.stream().count();                              // 5
nums.stream().anyMatch(n -> n > 4);                 // true
nums.stream().allMatch(n -> n > 0);                 // true
nums.stream().noneMatch(n -> n < 0);                // true

// reduce — fold a stream to a single value
int sum     = nums.stream().reduce(0, Integer::sum);          // 15
int product = nums.stream().reduce(1, (a, b) -> a * b);       // 120

// min / max — return Optional because the stream might be empty
nums.stream().max(Comparator.naturalOrder());       // Optional[5]

// findFirst — short-circuit, returns Optional
var first = nums.stream().filter(n -> n > 3).findFirst();
first.orElse(-1);                                    // 4

## Collectors

The `Collectors` utility class produces ready-made collectors for the common shapes. Pass them to `.collect(...)`.

In [ ]:
var words = List.of("alice", "bob", "carol", "dave", "eve");

// toList / toSet
List<String> lengths = words.stream().map(String::length).map(Object::toString).toList();
Set<Character> firstChars = words.stream().map(w -> w.charAt(0)).collect(Collectors.toSet());

// toMap — key extractor + value extractor
Map<String, Integer> byLength = words.stream()
    .collect(Collectors.toMap(w -> w, String::length));
// {alice=5, bob=3, carol=5, dave=4, eve=3}

// groupingBy — bucket by classifier
Map<Integer, List<String>> grouped = words.stream()
    .collect(Collectors.groupingBy(String::length));
// {3=[bob, eve], 4=[dave], 5=[alice, carol]}

// partitioningBy — boolean split into two buckets
Map<Boolean, List<String>> longShort = words.stream()
    .collect(Collectors.partitioningBy(w -> w.length() > 3));
// {false=[bob, eve], true=[alice, carol, dave]}

// joining — concatenate to a String
String joined = words.stream().collect(Collectors.joining(", ", "[", "]"));
// "[alice, bob, carol, dave, eve]"

// counting per group
Map<Integer, Long> countByLength = words.stream()
    .collect(Collectors.groupingBy(String::length, Collectors.counting()));
// {3=2, 4=1, 5=2}

## Parallel streams

Adding `.parallel()` (or starting from `collection.parallelStream()`) splits the work across the JVM's common ForkJoinPool. Each element is processed independently; the framework merges the partial results at the end.

Parallel streams pay off when:

- The dataset is large (typically >10,000 elements)
- The per-element work is non-trivial (not just incrementing)
- The operations are stateless and side-effect-free

They hurt when:

- The stream is small (overhead dominates)
- The collection has poor split characteristics (e.g., `LinkedList`)
- The lambdas mutate shared state (you'll get races)

When in doubt, **measure**. Don't default to `.parallel()`.

In [ ]:
// sequential
long sumSeq = LongStream.rangeClosed(1, 1_000_000).sum();          // 500000500000

// parallel — exact same answer, potentially faster on large inputs
long sumPar = LongStream.rangeClosed(1, 1_000_000).parallel().sum();

## `Optional<T>` — the type-safe alternative to `null`

`null` is famously *the billion-dollar mistake* — a value that compiles as any reference type but crashes the moment you dereference it. Java still has `null`, but for return values that might be absent, the modern signature is `Optional<T>`.

An `Optional<T>` is a small wrapper that either holds a value or is empty. The compiler forces you to acknowledge the empty case before reaching the inner value.

```text
   Optional<User>
       |
       +-- Optional[user]        present
       |
       +-- Optional.empty()      absent
```

In [ ]:
// creating
Optional<String> a = Optional.of("hello");         // never null — throws if you pass null
Optional<String> b = Optional.ofNullable(getMaybeNull());   // null becomes empty
Optional<String> c = Optional.empty();

// reading — never call .get() without checking
if (a.isPresent()) {
    System.out.println(a.get());
}

// better — ifPresent
a.ifPresent(System.out::println);

// even better — orElse / orElseGet / orElseThrow
String s1 = a.orElse("default");                            // returns "default" if empty
String s2 = a.orElseGet(() -> computeDefault());            // lazily compute fallback
String s3 = a.orElseThrow(() -> new IllegalStateException("missing"));

// transforming through the wrapper — map / filter / flatMap
Optional<Integer> len = a.map(String::length);              // Optional[5]
Optional<String> upper = a.filter(s -> s.length() > 3)
                          .map(String::toUpperCase);

**The Optional contract** — use it as a **return type** for values that may legitimately be absent. Do *not* use it for fields, method parameters, or collection elements. The convention exists because `Optional` is heavier than `null` and the type system already handles those cases (use a sensible default, an overload, or an empty collection instead).

## Streams + Optional together

The two were designed to compose. Stream terminal operations that might find nothing return `Optional` — `findFirst`, `findAny`, `min`, `max`, `reduce` without an identity.

In [ ]:
record User(String name, int age) {}

var users = List.of(
    new User("alice", 30),
    new User("bob", 25),
    new User("carol", 35)
);

// find a user — might not exist
Optional<User> youngest = users.stream().min(Comparator.comparingInt(User::age));
youngest.map(User::name).orElse("none");        // "bob"

// chained transformation
String result = users.stream()
    .filter(u -> u.age() > 100)
    .findFirst()
    .map(u -> "found " + u.name())
    .orElse("nobody over 100");
// "nobody over 100"